Inicialização

In [67]:
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re
import requests

driver = webdriver.Chrome()
driver.get("https://www.tum.de")

wait = WebDriverWait(driver, 15)

1. Teste do título:

In [33]:
def verifica_titulo():
    titulo = driver.title
    assert titulo == "The Entrepreneurial University - TUM"

2. Botão de retorno ao home - logo da universidade:

In [34]:
def retorna_home(lang):
    logo_superior = driver.find_element(By.CLASS_NAME, "page__header-logo")
    logo_superior.click()

    if lang == "DE":
        link_home = "https://www.tum.de/"
    else:
        link_home = "https://www.tum.de/en/"

    wait.until(EC.url_to_be(link_home))

3. Botão de mudança de língua:

In [35]:
def muda_lingua_home():
    bt_lingua = driver.find_element(By.CLASS_NAME, "page__header-language")
    elementos_li_lingua = bt_lingua.find_elements(By.TAG_NAME, "li")

    #muda de lingua
    try:
        li_link = elementos_li_lingua[1].find_element(By.TAG_NAME, "a") 
        lingua = li_link.text
        li_link.click()         
    except:
        li_link = elementos_li_lingua[0].find_element(By.TAG_NAME, "a")
        lingua = li_link.text
        li_link.click()

    #verifica se mudou de lingua
    wait.until(EC.text_to_be_present_in_element_attribute((By.CSS_SELECTOR, "html"), "lang", lingua.lower()))

    if lingua == "EN":
        print("Língua alterada para Inglês.")
    else:
        print("Língua alterada para Alemão")

    return lingua


4. Menu para alterar as configurações da tela (claro/escuro):

In [36]:
def muda_config():
    #Abertura de seleção: 
    bt_menu_configuracao = driver.find_element(By.XPATH, "/html/body/div[2]/header/div/div/div[2]/div[4]/button[1]")
    bt_menu_configuracao.click()
    ativo = bt_menu_configuracao.get_attribute("open")
    time.sleep(1)

    #Fecha seleção
    bt_fecha_menu_configuracao = driver.find_element(By.CSS_SELECTOR, "#theme-selector > form > button.btn.btn--dialog-close")
    bt_fecha_menu_configuracao.click()
    ativo = bt_menu_configuracao.get_attribute("open")
    time.sleep(1)

    #Abre novamente e testa os radios
    bt_menu_configuracao.click()
    campo_radio = driver.find_element(By.XPATH, "/html/body/div[2]/header/div/div/div[2]/div[4]/dialog/form/fieldset")
    radio_configuracao = campo_radio.find_elements(By.TAG_NAME, "input")
    for radio in radio_configuracao:
        time.sleep(1)
        if not radio.is_selected():
            radio.click()
        
    #Salva mudança na configuração da tela
    time.sleep(1)
    bt_menu_configuracao_salva = driver.find_element(By.XPATH, "/html/body/div[2]/header/div/div/div[2]/div[4]/dialog/form/button[2]") 
    bt_menu_configuracao_salva.click()

5. Botão para Pesquisa

In [37]:
def pesquisa_individual(valor = ""):
    campo_pesquisa = driver.find_element(By.ID, "search")
    campo_pesquisa.clear()
    campo_pesquisa.send_keys(valor)
    assert campo_pesquisa.get_attribute("value") == valor
    time.sleep(1)
    driver.find_element(By.CLASS_NAME, "btn.btn--search.search__submit").click()

    wait.until(EC.element_to_be_clickable((By.ID, "facettype")))
    fitros_resultados_pesq = driver.find_element(By.XPATH, "/html/body/div[2]/div/div/div[2]/div[1]")
    lista_filtros = fitros_resultados_pesq.find_elements(By.TAG_NAME, "li")
    if(not lista_filtros):
        print("Pesquisa sem filtros e resultados")
    else:
        print("Há ", len(lista_filtros), " filtros para essa pesquisa.")
    
    numero_resultados = driver.find_element(By.CSS_SELECTOR, "div[class='flex flex--space-between flex--middle flex--no-gutters'] div h2").text
    numero_resultados = re.sub(r'[^0-9]', '', numero_resultados)
    print("Há ", numero_resultados, " resultados para essa pesquisa.")
    

In [38]:
def controla_pesquisa():
    time.sleep(2)
    bt_pesquisa = driver.find_element(By.CLASS_NAME, "btn--search")
    bt_pesquisa.click()
    aba_pesquisa = driver.find_element(By.XPATH, "/html/body/div[2]/div")
    escondido = aba_pesquisa.get_attribute("aria-hidden")
    if escondido == "true":       #Verifica se nao esta aparecendo
        bt_pesquisa.click()     #clica de novo
    time.sleep(2)
    confirma_pesquisa = driver.find_element(By.XPATH, "/html/body/div[2]/div/div/div[1]/div/div[1]/div/div/div/button")
    
    #primeira pesquisa - sem resultados: se nao existe filtro ou se texto informa 0 resultados
    driver.find_element(By.ID, "search").clear()
    confirma_pesquisa.click()
    wait.until((
        lambda driver: driver.find_element(By.CSS_SELECTOR, ".search__resultset").find_elements(By.TAG_NAME, "div")
    ))
    fitros_resultados_pesq = driver.find_element(By.CSS_SELECTOR, ".search__filter")
    lista_filtros = fitros_resultados_pesq.find_elements(By.XPATH, ".//*")
    if not lista_filtros:
        print("Não há filtros e resultados da pesquisa")
    time.sleep(2)
    
    #segunda pesquisa
    pesquisa_individual("computer science")  #segunda pesquisa - com centenas de resultados

    time.sleep(5)
    bt_fecha_pesquisa = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "btn--close")))
    bt_fecha_pesquisa.click()

6. Botão que seleciona mais informações sobre a universidade com foco para grupos diferentes - Direciona para páginas diferentes

In [39]:
def mostra_navegacao():
    bt_navegacao_paginas = driver.find_element(By.CSS_SELECTOR, ".btn.btn--target-nav")
    aparece = bt_navegacao_paginas.get_attribute("aria-pressed")
    if aparece == "false":       #Verifica se esta aparecendo
        bt_navegacao_paginas.click()
        aparece = bt_navegacao_paginas.get_attribute("aria-pressed")
        assert aparece == "true"

def navega_paginas():
    time.sleep(2)
    mostra_navegacao()
    opcao_navegacao = driver.find_element(By.CLASS_NAME, "target-nav").find_elements(By.TAG_NAME, "li")
    classes_opcoes = []
    for opcao in opcao_navegacao:
        classes_opcoes.append(opcao.get_attribute("class"))

    for i in range(0, len(classes_opcoes)):
        mostra_navegacao()
        opcao = driver.find_element(By.CLASS_NAME, classes_opcoes[i])
        opcao.click()
        wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "c-target-header__content-text")))
        time.sleep(1)


7. Vídeo do Header em home

In [40]:
def verifica_video(caminho):
    video = driver.find_element(By.XPATH, caminho)

    esta_pausado = driver.execute_script("return arguments[0].paused;", video)
    if esta_pausado:
        driver.execute_script("arguments[0].play();", video)
    time.sleep(3)

    tempo_inicial = driver.execute_script("return arguments[0].currentTime;", video)
    time.sleep(2)

    tempo_atual = driver.execute_script("return arguments[0].currentTime;", video)

    if tempo_atual > tempo_inicial:
        print("O vídeo está sendo reproduzido normalmente.")
    else:
        print("O vídeo travou ou não está tocando.")
    time.sleep(2)


8. Botão para entrar site alternativo: School of Computation, Information and Technology

In [41]:
def abre_school_cit():
    quant_windows_inicial = len(driver.window_handles)
    schools = driver.find_element(By.CSS_SELECTOR, "nav[aria-labelledby='footer-schools']").find_elements(By.TAG_NAME, "li")
    link_cit = schools[0].find_element(By.TAG_NAME, "a")
    ActionChains(driver).scroll_to_element(schools[0]).perform()
    time.sleep(2)
    link_cit.click()
    
    #Verifica se existe e muda
    original_window_handle = driver.current_window_handle
    wait.until(EC.number_of_windows_to_be(quant_windows_inicial + 1))
    new_window_handle = (set(driver.window_handles) - {original_window_handle}).pop()
    driver.switch_to.window(new_window_handle)
    assert driver.current_window_handle == new_window_handle

9. Aceita somente cookies necessários

In [42]:
def aceita_cookies_neces():
    abre_cookies = ""       #somente se o botao nao esta escondido
    if "en" in driver.current_url:
        abre_cookies = driver.find_element(By.CSS_SELECTOR, "button[aria-label='revoke cookie consent']")
    else:
        abre_cookies = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Cookie Consent widerrufen']")
    
    if 'hide' in abre_cookies.__class__.__name__:
        cookies_neces = driver.find_element(By.CSS_SELECTOR, "button[class='cc-btn cc-deny']")
        cookies_neces.click()

10. Muda política de cookies

In [43]:
def muda_cookies(tipo):
    abre_cookies = ""       #somente se o botao nao esta escondido
    if "en" in driver.current_url:
        abre_cookies = driver.find_element(By.CSS_SELECTOR, "button[aria-label='revoke cookie consent']")
    else:
        abre_cookies = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Cookie Consent widerrufen']")

    cookies_classe = abre_cookies.get_attribute("class")
    if not 'hide' in cookies_classe:
        driver.execute_script("arguments[0].click();", abre_cookies)

    if tipo == "n":
        cookies_neces = wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/div/div/div[1]/button[1]")))
        cookies_neces.click()
    elif tipo == "a":
        all_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/div/div/div[1]/button[1]")))
        all_cookies.click()


11. Carrossel de Imagens

In [44]:
def verifica_carrossel_img():
    carrossel = driver.find_element(By.XPATH, "/html/body/main/div[2]/div/section")
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", carrossel)
    time.sleep(1)

    bt_prox_slide = driver.find_element(By.CLASS_NAME, "carousel-control-next")
    slides = driver.find_element(By.CLASS_NAME, "carousel-inner").find_elements(By.TAG_NAME, "div")
    slide_inicial = slides[0]
    for s in slides:
        if 'active' in s.get_attribute("class"):
            slide_inicial = s
    #slide_inicial = driver.find_element(By.CLASS_NAME, "carousel-item active")
    ActionChains(driver).move_to_element(bt_prox_slide).click().perform()
    slides = driver.find_element(By.CLASS_NAME, "carousel-inner").find_elements(By.TAG_NAME, "div")
    slide_atual = slides[0]
    for s in slides:
        if 'active' in s.get_attribute("class"):
            slide_atual = s
    assert slide_atual is not slide_inicial
    time.sleep(1)


12. Muda lingua em sites alternativos como aba da School CIT

In [45]:
def muda_lingua_cit():
    bt_lingua = driver.find_element(By.CLASS_NAME, "c-languagenav")
    elementos_li_lingua = bt_lingua.find_elements(By.TAG_NAME, "li")

    #muda de lingua
    try:
        li_link = elementos_li_lingua[1].find_element(By.TAG_NAME, "a") 
        lingua = li_link.get_attribute("hreflang")
        lingua = lingua[:2]
        driver.execute_script("arguments[0].click();", li_link)     
    except:
        li_link = elementos_li_lingua[0].find_element(By.TAG_NAME, "a")
        lingua = li_link.get_attribute("hreflang")
        lingua = lingua[:2]
        driver.execute_script("arguments[0].click();", li_link)     

    wait.until(EC.text_to_be_present_in_element_attribute((By.CSS_SELECTOR, "html"), "lang", lingua.lower()))

    if lingua == "en":
        print("Língua alterada para Inglês.")
    else:
        print("Língua alterada para Alemão")
    return lingua.upper()

13. Busca da School CIT com Google Search

In [ ]:
def pesquisa_cit_google():
    bt_pesquisa = driver.find_element(By.XPATH, "//button[@class='btn c-global-search__toggle js-search-toggle']")
    bt_pesquisa.click()

    mensagem_permissao_Google = wait.until(EC.visibility_of_element_located((By.ID, "global-search-overlay")))
    time.sleep(2)
    bt_confirma = driver.find_element(By.XPATH, "/html/body/main/dialog/form/button[1]")
    bt_confirma.click()

    valor = "computer science"
    campo_pesquisa = driver.find_element(By.ID, "cse-q")
    campo_pesquisa.clear()
    campo_pesquisa.send_keys(valor)
    assert campo_pesquisa.get_attribute("value") == valor
    time.sleep(1)
    driver.find_element(By.XPATH, "/html/body/header/div[1]/div/div/div[1]/div[2]/span/form/button").click()

    resultados = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "show-search")))
    if resultados.get_attribute("data-pid") == "14":        #se a pagina nao for alterada, data-pid = 6
        print("Aba com resultados da pesquisa foi aberta.")

14. Acesso a Home do site CIT pelo Menu

In [47]:
def acesso_menu_cit():
    #se esta na pagina de pesquisa de cit
    if 'cit.tum.de' in driver.current_url and driver.find_element(By.CLASS_NAME, "show-search").get_attribute("data-pid") == "14": 
        lingua = ''
        caminho_home_cit = driver.find_element(By.ID, "sitenav")
        if "en" in driver.current_url[:30]:     #para evitar que a pesquisa do google tenha "en"
            lingua = 'en'
            home_cit = caminho_home_cit.find_element(By.CSS_SELECTOR, "a[title='Home']")
        else:
            lingua = 'de'
            home_cit = caminho_home_cit.find_element(By.CSS_SELECTOR, "a[title='Startseite']")
        time.sleep(1)
        
        #teste dos menus (desdobramento)
        menu_cit = driver.find_element(By.ID, "menuMain").find_elements(By.CLASS_NAME, "accordion-item.c-sitenav__item")
        if lingua == 'en':
            bt_studies = menu_cit[0].find_element(By.CSS_SELECTOR, "button[aria-label='Open submenu of Studies']")
        else:
            bt_studies = menu_cit[0].find_element(By.CSS_SELECTOR, "button[aria-label='Öffne das Untermenü von Studium']")
        #bt_studies = menu_cit[0].find_element(By.XPATH, "//button[contains(@class,'accordion-button c-sitenav__button')]")  #pode ser collapse ou nao
        bt_studies.click()
        wait.until(EC.text_to_be_present_in_element_attribute((By.ID, "menuMain1"), "class", "show"))
        
        #retorna home
        time.sleep(3)
        home_cit.click()

15. Retorna ao Home da Universidade

In [ ]:
def retorna_home_de_cit():
    #if 'home' in driver.current_url[:30] or 'startseite' in driver.current_url[:30]:
    div_home = driver.find_element(By.CSS_SELECTOR, "div[class='container c-siteorg__grid'] div:nth-child(2)")
    bt_home = div_home.find_element(By.TAG_NAME, "a")
    bt_home.click()
    time.sleep(3)

16. Acesso às Redes Sociais

In [ ]:
def redes_sociais():
    redes_sociais = driver.find_element(By.CSS_SELECTOR, "nav[aria-label='Social Media']")
    ActionChains(driver).scroll_to_element(redes_sociais).perform()
    time.sleep(2)

    elemento_links_redes = redes_sociais.find_elements(By.TAG_NAME, "a")
    links_redes = []
    for e in elemento_links_redes:
        links_redes.append(e.get_attribute("href"))

    nomes_redes = ["linkedin", "instagram", "bsky", "wisskomm", "threads", "facebook", "youtube", "x.com", "news.rss"]
    nomes_redes_val = []

    for link in links_redes:
        if not link or link.startswith("javascript:"):      #links vazios ou nulo
            continue

        try:
            resposta = requests.get(link, timeout=5)
            if resposta.status_code >= 400:         #indicam erro (pagina nao encontrada, acesso negado, etc.)
                print(f"[Link quebrado ou inacessível] {link} - {resposta.status_code}") 
            else:       #adiciona nomes validos
                for n in nomes_redes:
                    if n in link:
                        if n == "news.rss":
                            nomes_redes_val.append("Notícias TUM")
                        else:
                            nomes_redes_val.append(n)
        except requests.exceptions.RequestException as e:       #erros de conexao, tempo esgotado (timeout)
            print(f"[Erro de conexão no link] {link} - Erro: {e}") 

    print("Redes Sociais Disponíveis: ")
    print(*nomes_redes_val, sep = ', ')

17. Acesso à Página de FeedBack

In [50]:
def acessa_feedback():
    navegacao_outros = driver.find_element(By.CSS_SELECTOR, "nav[aria-label='Meta-Navigation']")
    ActionChains(driver).scroll_to_element(navegacao_outros).perform()
    bt_feedback = navegacao_outros.find_elements(By.TAG_NAME, "a")
    time.sleep(2)
    bt_feedback[1].click()

    if not 'tum' in driver.current_url and 'feedback' in driver.current_url:
        print("Não foi possível acessar o site.")
    wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "c-header__content")))
    time.sleep(2)

18. Preenche Campos do Feedback

In [51]:
def preenche_feedback():
    formulario = driver.find_element(By.CSS_SELECTOR, ".fieldset.powermail_fieldset.powermail_fieldset_13")
    ActionChains(driver).scroll_to_element(formulario).perform()
    time.sleep(2)
    print("Preenchimento do feedback: ")

    campo_assunto = formulario.find_element(By.ID, "powermail_field_marker_01")
    campo_assunto.clear()
    valor = "Test with Selenium at Page Jobs"
    campo_assunto.send_keys(valor)
    assert campo_assunto.get_attribute("value") == valor
    print("Assunto: ", valor)
    time.sleep(2)

    campo_feedback = formulario.find_element(By.ID, "powermail_field_marker_02")
    campo_feedback.clear()
    valor = "While conducting a test with Selenium in Software Quality course at the Federal University of Santa Maria (UFSM), " \
        "it was noticed that it is not possible to use the Subscribe to Job Announcements (RSS) link/file."
    campo_feedback.send_keys(valor)
    assert campo_feedback.get_attribute("value") == valor
    print("Mensagem: ", valor)
    time.sleep(5)    

19. Aceita Coleta de Dados e envia o Formulário

In [ ]:
def checkbox_permite():
    formulario = driver.find_element(By.CSS_SELECTOR, ".fieldset.powermail_fieldset.powermail_fieldset_13")
    permissao = formulario.find_element(By.ID, "powermail_field_marker_03_1")
    ActionChains(driver).scroll_to_element(permissao).perform()
    time.sleep(2)
    
    if not permissao.is_selected():
        permissao.click()
        assert permissao.is_selected()
    
    time.sleep(3)

    # bt_envio_formulario = formulario.find_element(By.CSS_SELECTOR, "button[class='btn btn--primary']")
    # ActionChains(driver).scroll_to_element(bt_envio_formulario).perform()
    # bt_envio_formulario.click()

20. Acesso à Página de Vagas de Trabalhos na TUM

In [53]:
def acessa_vagas_trabalhos():
    navegacao_outros = driver.find_element(By.CSS_SELECTOR, "nav[aria-label='Meta-Navigation']")
    ActionChains(driver).scroll_to_element(navegacao_outros).perform()
    bt_jobs = navegacao_outros.find_element(By.TAG_NAME, "a")
    time.sleep(2)
    bt_jobs.click()

    if not (driver.current_url == "https://www.tum.de/ueber-die-tum/karriere-und-jobs/stellenangebote" 
        or driver.current_url == "https://www.tum.de/en/about-tum/careers-and-jobs/careers-at-tum"):
        print("Não foi possível acessar o site.")
    wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "c-header__content")))

    subscribe_job_announcements = driver.find_element(By.CLASS_NAME, "ce-textmedia__content")
    ActionChains(driver).scroll_to_element(subscribe_job_announcements).perform()
    bt_subscribe_job = driver.find_element(By.TAG_NAME, "a")
    if bt_subscribe_job:
        bt_subscribe_job.click()

Ordem dos testes das ações

In [ ]:
time.sleep(5)
verifica_titulo()                                   #1
lingua = muda_lingua_home()                         #2
retorna_home(lingua)                                #3
muda_config()                                       #4
controla_pesquisa()                                 #5
navega_paginas()                                    #6
retorna_home(lingua)
verifica_video("//video[@autoplay='autoplay']")     #7
abre_school_cit()                                   #8
aceita_cookies_neces()                              #9
muda_cookies("a")                                   #10
muda_cookies("n")                                   
verifica_carrossel_img()                            #11
lingua = muda_lingua_cit()                          #12
pesquisa_cit_google()                               #13
acesso_menu_cit()                                   #14
retorna_home_de_cit()                               #15
redes_sociais()                                     #16
acessa_feedback()                                   #17
preenche_feedback()                                 #18
checkbox_permite()                                  #19
retorna_home(lingua)
acessa_vagas_trabalhos()                            #20

Língua alterada para Inglês.
Há  5  filtros para essa pesquisa.
Há  745  resultados para essa pesquisa.
O vídeo está sendo reproduzido normalmente.
Língua alterada para Alemão
Aba com resultados da pesquisa foi aberta.
Redes Sociais Disponíveis: 
linkedin, instagram, bsky, wisskomm, threads, facebook, youtube, x.com, Notícias TUM
Preenchimento do feedback: 
Assunto:  Test with Selenium at Page Jobs
Mensagem:  While conducting a test with Selenium in Software Quality course at the Federal University of Santa Maria (UFSM), it was noticed that it is not possible to use the Subscribe to Job Announcements (RSS) link/file.


Finalizando

In [69]:
quant_windows_inicial = len(driver.window_handles)
quant = quant_windows_inicial
while quant > 1:
    driver.switch_to.window(driver.window_handles[quant-1])
    driver.close()
    quant = len(driver.window_handles)
    time.sleep(0.5)

In [70]:
driver.quit()